# Analysis Notebook — experiment_v4
Carrega resultats, winsoritza, genera figures i exporta CSV+HTML.


In [ ]:
import os, json
from datetime import datetime
import numpy as np
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

OUT_DIR      = os.path.join(os.getcwd(), 'outputs', 'experiments')
RESULTS_FILE = os.path.join(OUT_DIR, 'experiment_results.json')
CSV_OUT      = os.path.join(OUT_DIR, 'experiment_table.csv')
PNG_OUT      = os.path.join(OUT_DIR, 'analysis_summary.png')
HTML_OUT     = os.path.join(OUT_DIR, 'analysis_summary.html')
os.makedirs(OUT_DIR, exist_ok=True)
print(f'OUT_DIR: {OUT_DIR}')


In [ ]:
runs = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        runs = json.load(f).get('runs', [])
if not runs:
    for fn in sorted(os.listdir(OUT_DIR)):
        if fn.startswith('run_') and fn.endswith('.json'):
            try:
                with open(os.path.join(OUT_DIR, fn)) as f:
                    runs.append(json.load(f))
            except: pass
print(f'Runs: {len(runs)}')


In [ ]:
rows = []
for r in runs:
    m  = r.get('metrics') or {}
    px = m.get('policy_x', [])
    rows.append({
        'run_id':r.get('run_id'), 'phase':r.get('phase'),
        'seed':r.get('seed'),    'status':r.get('status'),
        'runtime_s':r.get('runtime_s'),
        'dP_mean':m.get('dP_mean'),  'P_std':m.get('P_std'),
        'r_mean':m.get('r_mean'),    'tau_var':m.get('tau_var'),
        'J_mean':m.get('J_mean'),    'J_final':m.get('J_final'),
        'K_bus_fin':m.get('K_bus_fin'), 'K_ind_fin':m.get('K_ind_fin'),
        'E_cicle':m.get('E_cicle'),
        'via1_MJ':m.get('via1_MJ'),  'via2_MJ':m.get('via2_MJ'),
        'via3_MJ':m.get('via3_MJ'),  'shield_MJ':m.get('shield_MJ'),
        'p_win':px[0] if len(px)>0 else None,
        'm_exc_n':px[1] if len(px)>1 else None,
    })
df = pd.DataFrame(rows)
def winsorize(s, lo=5, hi=95):
    v = s.dropna()
    if len(v)<4: return s
    return s.clip(np.percentile(v,lo), np.percentile(v,hi))
df['dP_w'] = df.groupby('phase')['dP_mean'].transform(winsorize)
df.to_csv(CSV_OUT, index=False)
print('CSV:', CSV_OUT)
df.groupby('phase')[['dP_mean','dP_w','r_mean','tau_var']].describe().round(4)


In [ ]:
def phase_stats(phase):
    sub = df[(df.phase==phase) & (df.status=='ok')]
    if sub.empty: return {}
    dps = sub.dP_w.dropna()
    return {
        'n': len(sub),
        'dP_mean': round(float(dps.mean()),3),
        'dP_std':  round(float(dps.std()),3),
        'pct_pos': round(float(100*(dps>0).sum()/len(dps)),1),
        'tau_mean':round(float(sub.tau_var.mean()),4),
        'r_mean':  round(float(sub.r_mean.mean()),4),
    }
summary = {p: phase_stats(p) for p in df.phase.unique()}
print(json.dumps(summary, indent=2))


In [ ]:
BG='#161b22'; TXT='#c9d1d9'
CG='#97C459'; CR='#ED93B1'; CQ='#FAC775'
CV='#5DCAA5'; CM='#AFA9EC'

def style(ax, title, xl='', yl=''):
    ax.set_facecolor(BG); ax.set_title(title, color=TXT, fontsize=9)
    if xl: ax.set_xlabel(xl, color=TXT, fontsize=8)
    if yl: ax.set_ylabel(yl, color=TXT, fontsize=8)
    ax.tick_params(colors=TXT, labelsize=7)
    for sp in ax.spines.values(): sp.set_color('#30363d')
    ax.grid(True, color='#21262d', linewidth=0.4)

fig, axes = plt.subplots(2, 3, figsize=(16,10))
fig.patch.set_facecolor('#0d1117')

base = df[(df.phase=='baseline') & (df.status=='ok')]
mc_d = df[(df.phase=='montecarlo') & (df.status=='ok')]
pol  = df[(df.phase=='policy') & (df.status=='ok')]
ab   = df[df.phase.str.startswith('ablation') & (df.status=='ok')]

ax = axes[0,0]
if not base.empty:
    v = base.dP_w.values
    ax.bar(range(len(v)), v, color=[CG if x>0 else CR for x in v], width=0.7, alpha=0.8)
    ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(float(v.mean()), color=CQ, lw=1.5, ls='--', label=f'mitj={v.mean():+.2f}')
    ax.legend(fontsize=7, facecolor=BG, edgecolor='#30363d', labelcolor=TXT)
style(ax, 'Baseline ΔP (winsoritzat)', 'seed', 'ΔP [MW]')

ax = axes[0,1]
if not mc_d.empty:
    mc_d.dP_w.hist(ax=ax, bins=16, color=CV, alpha=0.8, edgecolor='#30363d')
    ax.axvline(float(mc_d.dP_w.mean()), color=CQ, lw=1.5, ls='--')
    ax.axvline(0, color='white', lw=0.8, ls='--', alpha=0.5)
    pct = 100*(mc_d.dP_w>0).mean()
    ax.text(0.97,0.95,f'{pct:.0f}%>0',transform=ax.transAxes,
            ha='right',va='top',color=CG if pct>50 else CR,fontsize=9)
style(ax, 'MC ΔP distribució', 'ΔP [MW]', 'N')

ax = axes[0,2]
if not pol.empty:
    ax.scatter(pol.dP_w, pol.J_final, color=CM, s=60)
    ax.axvline(0, color='white', lw=0.5, ls='--', alpha=0.4)
style(ax, 'Política ΔP vs J_final', 'ΔP [MW]', 'J')

ax = axes[1,0]
if not ab.empty:
    g = ab.groupby('phase').dP_mean.mean().sort_values()
    g.plot(kind='bar', ax=ax, color=[CG if v>0 else CR for v in g.values], alpha=0.8)
    ax.axhline(0, color='white', lw=0.8, ls='--', alpha=0.5)
    ax.tick_params(axis='x', labelrotation=12)
style(ax, 'Ablation ΔP per config.', '', 'ΔP [MW]')

ax = axes[1,1]
if not base.empty:
    vies = [('via1_MJ','Shear',CG),('via2_MJ','Grav.',CR),
            ('via3_MJ','Res.',CM),('shield_MJ','Escut',CQ)]
    for col,label,color in vies:
        if col in base.columns:
            ax.bar([label],[float(base[col].mean())],color=color,alpha=0.8,width=0.5)
    ax.axhline(0,color='white',lw=0.8,ls='--',alpha=0.5)
style(ax, 'Vies energia (mitja baseline)', '', 'MJ')

ax = axes[1,2]
if not df.empty:
    r_g = df[df.status=='ok'].groupby('phase').r_mean.mean().sort_values(ascending=False)
    r_g.plot(kind='bar', ax=ax, color=CV, alpha=0.8)
    ax.tick_params(axis='x', labelrotation=12)
style(ax, 'Ordre Kuramoto r per fase', '', 'r')

plt.tight_layout()
fig.savefig(PNG_OUT, dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
print('PNG:', PNG_OUT)


In [ ]:
html = (
    '<!DOCTYPE html><html><head>'
    '<meta charset=utf-8><title>Experiment v4</title>'
    '<style>body{font-family:monospace;background:#0d1117;color:#c9d1d9}'
    'pre{background:#161b22;padding:1rem;border-radius:8px}</style></head><body>'
    f'<h1>Experiment v4</h1>'
    f'<p>Generat: {datetime.utcnow().isoformat()}</p>'
    f'<h2>Estadistica</h2><pre>{json.dumps(summary, indent=2)}</pre>'
    f'<h2>Figures</h2><img src="{os.path.basename(PNG_OUT)}" width="100%">'
    '</body></html>'
)
with open(HTML_OUT, 'w') as f:
    f.write(html)
print('HTML:', HTML_OUT)


## Notes
- Ajusta `OUT_DIR` si executes des d'un directori diferent.
- Si pandas no esta disponible, substitueix el DataFrame per dicts de numpy.
- Backend `Agg` per funcionar en servidors headless.
